In [1]:
import pandas as pd

In [2]:
hor_2024_10_27 = pd.read_excel("../data/raw/election_results/October 27 2024 House of Representatives election.xls", skiprows = 3)

In [3]:
hor_2024_10_27.head()

,比例代表区,区分,自由民主党,立憲民主党,日本維新の会,公明党,日本共産党,国民民主党
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,北海道,北海道,"641,127","694,578.943","96,954","253,344","169,799","192,303.871"
3,東北,青森県,"175,840","128,030","19,020","46,357","34,182","53,931.960"
4,NaN,岩手県,"144,856","173,405.421","21,963","42,404","33,300","44,419.547"


In [4]:
#select only the columns with vote totals and skip empty rows 
df_2024 = hor_2024_10_27.iloc[2:,1:]
#rows contain groupings of vote totals (計) so skip those
df_2024 = df_2024.query("区分 != '計'")

In [5]:
#dataframe is stacked together in pages so separate these out before merging
page_1_2024 = df_2024.iloc[0:47,:]
page_2_2024 = df_2024.iloc[52:99,:]
page_3_2024 = df_2024.iloc[104:151,:2]

page_2_2024.columns = list(df_2024.iloc[49])
page_3_2024.columns = list(df_2024.iloc[101][0:2])


In [6]:
page_1_2024.tail()

,区分,自由民主党,立憲民主党,日本維新の会,公明党,日本共産党,国民民主党
52,熊本県,"241,040","140,133.299","41,516","100,854","24,694","54,060.654"
53,大分県,"141,626","116,150.325","22,352","69,848","22,303","40,688.656"
54,宮崎県,"139,807","84,958.631","24,267","63,485","16,563","45,507.346"
55,鹿児島県,"237,968","145,806.891","38,226","84,747","23,323","48,693.066"
56,沖縄県,"109,657","98,432.785","33,583","95,707","46,645","39,174.174"


In [7]:
vote_totals_2024 = pd.merge(page_1_2024, page_2_2024, on = '区分')
vote_totals_2024 = pd.merge(vote_totals_2024, page_3_2024, on = '区分')

In [8]:
vote_totals_2024.head()

,区分,自由民主党,立憲民主党,日本維新の会,公明党,日本共産党,国民民主党,れいわ新選組,社会民主党,参政党,みんなでつくる党,安楽死制度を考える会,日本保守党,合計
0,北海道,"641,127","694,578.943","96,954","253,344","169,799","192,303.871","177,620","31,134","57,002",NaN,"18,455","61,903","2,394,220.814"
1,青森県,"175,840","128,030","19,020","46,357","34,182","53,931.960","36,933","9,036","13,136",NaN,NaN,NaN,"516,465.960"
2,岩手県,"144,856","173,405.421","21,963","42,404","33,300","44,419.547","39,199","13,764","18,274",NaN,NaN,NaN,"531,584.968"
3,宮城県,"275,135","249,392.988","60,722","103,542","56,060","103,186.973","73,055","16,703","26,677",NaN,NaN,NaN,"964,473.961"
4,秋田県,"169,066","100,995.106","20,441","45,495","20,738","64,350.870","25,156","8,143","7,883",NaN,NaN,NaN,"462,267.976"


In [9]:
###TO DO: replace commas, chang NaN to 0, change to float, calculate percentages

#get rid of commas and NaN in data
vote_totals_2024 = vote_totals_2024.replace(',', '', regex=True)
vote_totals_2024 = vote_totals_2024.fillna(0)

#change 区分 to index
vote_totals_2024 = vote_totals_2024.set_index('区分')
vote_totals_2024.index.name = 'prefecture'

In [10]:
#change vote totals to floats
vote_totals_2024 = vote_totals_2024.astype(float)
#make new dataframe for percentages 
vote_percent_2024 = vote_totals_2024.copy()
for column in vote_percent_2024.columns:
    vote_percent_2024[column] = vote_percent_2024[column] / vote_totals_2024['合計']

vote_percent_2024.head()


,自由民主党,立憲民主党,日本維新の会,公明党,日本共産党,国民民主党,れいわ新選組,社会民主党,参政党,みんなでつくる党,安楽死制度を考える会,日本保守党,合計
prefecture,,,,,,,,,,,,,
北海道,0.267781,0.290106,0.040495,0.105815,0.070920,0.080320,0.074187,0.013004,0.023808,0.0,0.007708,0.025855,1.0
青森県,0.340468,0.247896,0.036827,0.089758,0.066184,0.104425,0.071511,0.017496,0.025434,0.0,0.000000,0.000000,1.0
岩手県,0.272498,0.326205,0.041316,0.079769,0.062643,0.083561,0.073740,0.025892,0.034376,0.0,0.000000,0.000000,1.0
宮城県,0.285269,0.258579,0.062959,0.107356,0.058125,0.106988,0.075746,0.017318,0.027660,0.0,0.000000,0.000000,1.0
秋田県,0.365732,0.218477,0.044219,0.098417,0.044861,0.139207,0.054419,0.017615,0.017053,0.0,0.000000,0.000000,1.0


In [11]:
from jp_eng_dicts import *

In [12]:
eng_vote_percent_2024 = vote_percent_2024.copy()
eng_vote_percent_2024 = eng_vote_percent_2024.rename(columns = jp_party_to_eng)
eng_vote_percent_2024 = eng_vote_percent_2024.rename(columns = {'合計' : 'total'})
eng_vote_percent_2024 = eng_vote_percent_2024.rename(index = jp_prefect_to_eng)

In [13]:
eng_vote_percent_2024.head()

,Liberal Democratic Party,Constitutional Democratic Party,Japan Innovation Party,Komeito,Japanese Communist Party,Democratic Party For the People,Reiwa Shinsengumi,Social Democratic Party,Sanseito,Minna De Tsukuru Party,Euthanasia Consideration Party,Conservative Party of Japan,total
prefecture,,,,,,,,,,,,,
Hokkaido,0.267781,0.290106,0.040495,0.105815,0.070920,0.080320,0.074187,0.013004,0.023808,0.0,0.007708,0.025855,1.0
Aomori,0.340468,0.247896,0.036827,0.089758,0.066184,0.104425,0.071511,0.017496,0.025434,0.0,0.000000,0.000000,1.0
Iwate,0.272498,0.326205,0.041316,0.079769,0.062643,0.083561,0.073740,0.025892,0.034376,0.0,0.000000,0.000000,1.0
Miyagi,0.285269,0.258579,0.062959,0.107356,0.058125,0.106988,0.075746,0.017318,0.027660,0.0,0.000000,0.000000,1.0
Akita,0.365732,0.218477,0.044219,0.098417,0.044861,0.139207,0.054419,0.017615,0.017053,0.0,0.000000,0.000000,1.0


In [14]:
eng_vote_total_2024 = vote_totals_2024.copy()
eng_vote_total_2024 = eng_vote_total_2024.rename(columns = jp_party_to_eng)
eng_vote_total_2024 = eng_vote_total_2024.rename(columns = {'合計' : 'total'})
eng_vote_total_2024 = eng_vote_total_2024.rename(index = jp_prefect_to_eng)

In [15]:
eng_vote_total_2024.head(3)

,Liberal Democratic Party,Constitutional Democratic Party,Japan Innovation Party,Komeito,Japanese Communist Party,Democratic Party For the People,Reiwa Shinsengumi,Social Democratic Party,Sanseito,Minna De Tsukuru Party,Euthanasia Consideration Party,Conservative Party of Japan,total
prefecture,,,,,,,,,,,,,
Hokkaido,641127.0,694578.943,96954.0,253344.0,169799.0,192303.871,177620.0,31134.0,57002.0,0.0,18455.0,61903.0,2394220.814
Aomori,175840.0,128030.000,19020.0,46357.0,34182.0,53931.960,36933.0,9036.0,13136.0,0.0,0.0,0.0,516465.960
Iwate,144856.0,173405.421,21963.0,42404.0,33300.0,44419.547,39199.0,13764.0,18274.0,0.0,0.0,0.0,531584.968


In [16]:
eng_vote_total_2024.to_csv("../data/clean/election_results/house_of_representatives_20241027_vote_totals")
eng_vote_percent_2024.to_csv("../data/clean/election_results/house_of_representatives_20241027_vote_percent")


In [17]:
hor_2026_02_08 = pd.read_excel("../data/raw/election_results/February 8 2026 House of Representatives election.xlsx", skiprows = 3)

In [18]:
hor_2026_02_08.head()

,選挙区,都道府県,自由民主党,中道改革連合,日本維新の会,国民民主党,参政党,日本共産党
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,北海道,北海道,911742,605889,93966,218850,163329,134084
2,東北,青森県,207510,111047,17605,41857,32441,21576
3,NaN,岩手県,218970,126468,21286,56518,41841,27502
4,NaN,宮城県,407292,227663,50301,86489,84169,41769


In [19]:
df_2026 = hor_2026_02_08.iloc[1:,1:]
df_2026 = df_2026.query("都道府県 != '計'")
df_2026.tail()

,都道府県,自由民主党,中道改革連合,日本維新の会,国民民主党,参政党,日本共産党
172,大分県,525121,NaN,NaN,NaN,NaN,NaN
173,宮崎県,455856,NaN,NaN,NaN,NaN,NaN
174,鹿児島県,697082,NaN,NaN,NaN,NaN,NaN
175,沖縄県,628572,NaN,NaN,NaN,NaN,NaN
177,合計,57259957,NaN,NaN,NaN,NaN,NaN


In [20]:
page_1_2026 = df_2026.iloc[0:47,:]
page_2_2026 = df_2026.iloc[51:98,:]
page_3_2026 = df_2026.iloc[102:149, :2]

page_2_2026.columns = df_2026.iloc[49]
page_3_2026.columns = df_2026.iloc[100][0:2]

vote_totals_2026 = pd.merge(page_1_2026, page_2_2026, on = '都道府県')
vote_totals_2026 = pd.merge(vote_totals_2026, page_3_2026, on = '都道府県')

vote_totals_2026 = vote_totals_2026.set_index('都道府県')
vote_totals_2026 = vote_totals_2026.fillna(0)
vote_totals_2026.index.name = 'prefecture'
vote_totals_2026 = vote_totals_2026.astype(float)


In [21]:
vote_totals_2026.head()

,自由民主党,中道改革連合,日本維新の会,国民民主党,参政党,日本共産党,れいわ新選組,減税日本・ゆうこく連合,日本保守党,社会民主党,チームみらい,安楽死制度を考える会,合計
prefecture,,,,,,,,,,,,,
北海道,911742.0,605889.0,93966.0,218850.0,163329.0,134084.0,76099.0,32878.0,60119.0,31754.0,134613.0,0.0,2463323.0
青森県,207510.0,111047.0,17605.0,41857.0,32441.0,21576.0,17285.0,0.0,10166.0,7609.0,25042.0,0.0,492138.0
岩手県,218970.0,126468.0,21286.0,56518.0,41841.0,27502.0,22328.0,0.0,10008.0,11027.0,33081.0,0.0,569029.0
宮城県,407292.0,227663.0,50301.0,86489.0,84169.0,41769.0,29141.0,0.0,22041.0,11602.0,81363.0,0.0,1041830.0
秋田県,196090.0,75552.0,24424.0,65635.0,23402.0,15087.0,10592.0,0.0,6831.0,5982.0,19299.0,0.0,442894.0


In [22]:
vote_percent_2026 = vote_totals_2026.copy()
for column in vote_percent_2026.columns:
    vote_percent_2026[column] = vote_percent_2026[column] / vote_totals_2026['合計']

vote_percent_2026 = vote_percent_2026.rename(columns = jp_party_to_eng)
vote_percent_2026 = vote_percent_2026.rename(columns = {'合計':'total'})
vote_percent_2026 = vote_percent_2026.rename(index = jp_prefect_to_eng)



In [23]:
vote_percent_2026.head()

,Liberal Democratic Party,Centrist Reform Alliance,Japan Innovation Party,Democratic Party For the People,Sanseito,Japanese Communist Party,Reiwa Shinsengumi,Tax Cuts Japan and Yukoku Alliance,Conservative Party of Japan,Social Democratic Party,Team Mirai,Euthanasia Consideration Party,total
prefecture,,,,,,,,,,,,,
Hokkaido,0.370127,0.245964,0.038146,0.088843,0.066304,0.054432,0.030893,0.013347,0.024406,0.012891,0.054647,0.0,1.0
Aomori,0.421650,0.225642,0.035772,0.085051,0.065919,0.043841,0.035122,0.000000,0.020657,0.015461,0.050884,0.0,1.0
Iwate,0.384813,0.222252,0.037408,0.099324,0.073531,0.048331,0.039239,0.000000,0.017588,0.019379,0.058136,0.0,1.0
Miyagi,0.390939,0.218522,0.048281,0.083016,0.080790,0.040092,0.027971,0.000000,0.021156,0.011136,0.078096,0.0,1.0
Akita,0.442747,0.170587,0.055146,0.148196,0.052839,0.034065,0.023915,0.000000,0.015424,0.013507,0.043575,0.0,1.0


In [25]:
vote_totals_2026.to_csv("../data/clean/election_results/house_of_representatives_20260208_vote_totals")
vote_percent_2026.to_csv("../data/clean/election_results/house_of_representatives_20260208_vote_percent")